# Synthetic Data Generation Using RAGAS - RAG Evaluation with LangSmith

In the following notebook we'll explore a use-case for RAGAS' synthetic testset generation workflow!



- 🤝 BREAKOUT ROOM #1
  1. Use RAGAS to Generate Synthetic Data

- 🤝 BREAKOUT ROOM #2
  1. Load them into a LangSmith Dataset
  2. Evaluate our RAG chain against the synthetic test data
  3. Make changes to our pipeline
  4. Evaluate the modified pipeline

SDG is a critical piece of the puzzle, especially for early iteration! Without it, it would not be nearly as easy to get high quality early signal for our application's performance.

Let's dive in!

# 🤝 BREAKOUT ROOM #1

## Task 1: Dependencies and API Keys

We'll need to install a number of API keys and dependencies, since we'll be leveraging a number of great technologies for this pipeline!

1. OpenAI's endpoints to handle the Synthetic Data Generation
2. OpenAI's Endpoints for our RAG pipeline and LangSmith evaluation
3. QDrant as our vectorstore
4. LangSmith for our evaluation coordinator!

Let's install and provide all the required information below!

## Dependencies and API Keys:

> NOTE: DO NOT RUN THESE CELLS IF YOU ARE RUNNING THIS NOTEBOOK LOCALLY

In [2]:
#!pip install -qU ragas==0.2.10

In [3]:
#!pip install -qU langchain-community==0.3.14 langchain-openai==0.2.14 unstructured==0.16.12 langgraph==0.2.61 langchain-qdrant==0.2.0

### NLTK Import

To prevent errors that may occur based on OS - we'll import NLTK and download the needed packages to ensure correct handling of data.

In [4]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/santhoshchaka/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/santhoshchaka/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [5]:
import os
import getpass

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")

We'll also want to set a project name to make things easier for ourselves.

In [6]:
from uuid import uuid4

os.environ["LANGCHAIN_PROJECT"] = f"AIM - SDG - {uuid4().hex[0:8]}"

OpenAI's API Key!

In [7]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

## Generating Synthetic Test Data

We wil be using Ragas to build out a set of synthetic test questions, references, and reference contexts. This is useful because it will allow us to find out how our system is performing.

> NOTE: Ragas is best suited for finding *directional* changes in your LLM-based systems. The absolute scores aren't comparable in a vacuum.

### Data Preparation

We'll prepare our data - which should hopefull be familiar at this point since it's our Loan Data use-case!

Next, let's load our data into a familiar LangChain format using the `DirectoryLoader`.

In [8]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader


path = "data/"
loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()

### Knowledge Graph Based Synthetic Generation

Ragas uses a knowledge graph based approach to create data. This is extremely useful as it allows us to create complex queries rather simply. The additional testset complexity allows us to evaluate larger problems more effectively, as systems tend to be very strong on simple evaluation tasks.

Let's start by defining our `generator_llm` (which will generate our questions, summaries, and more), and our `generator_embeddings` which will be useful in building our graph.

### Unrolled SDG

In [10]:
import sys
print("Notebook is using Python at:", sys.executable)
!{sys.executable} -m pip install pandas

Notebook is using Python at: /opt/homebrew/opt/python@3.10/bin/python3.10
  Using cached pandas-2.3.1-cp310-cp310-macosx_11_0_arm64.whl.metadata (91 kB)
  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2025.2-py2.py3-none-any.whl.metadata (1.4 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 76.2 MB/s eta 0:00:00
Using cached pytz-2025.2-py2.py3-none-any.whl (509 kB)
Using cached tzdata-2025.2-py2.py3-none-any.whl (347 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [pandas]2m2/3 [pandas]


In [12]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

In [13]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/santhoshchaka/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/santhoshchaka/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

Next, we're going to instantiate our Knowledge Graph.

This graph will contain N number of nodes that have M number of relationships. These nodes and relationships (AKA "edges") will define our knowledge graph and be used later to construct relevant questions and responses.

In [16]:
import sys
print("Notebook is using Python at:", sys.executable)
!{sys.executable} -m pip install pillow

Notebook is using Python at: /opt/homebrew/opt/python@3.10/bin/python3.10
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 62.8 MB/s eta 0:00:00


In [17]:
from ragas.testset.graph import KnowledgeGraph

kg = KnowledgeGraph()
kg

KnowledgeGraph(nodes: 0, relationships: 0)

The first step we're going to take is to simply insert each of our full documents into the graph. This will provide a base that we can apply transformations to.

In [18]:
from ragas.testset.graph import Node, NodeType

### NOTICE: We're using a subset of the data for this example - this is to keep costs/time down.
for doc in docs[:20]:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )
kg

KnowledgeGraph(nodes: 20, relationships: 0)

Now, we'll apply the *default* transformations to our knowledge graph. This will take the nodes currently on the graph and transform them based on a set of [default transformations](https://docs.ragas.io/en/latest/references/transforms/#ragas.testset.transforms.default_transforms).

These default transformations are dependent on the corpus length, in our case:

- Producing Summaries -> produces summaries of the documents
- Extracting Headlines -> finding the overall headline for the document
- Theme Extractor -> extracts broad themes about the documents

It then uses cosine-similarity and heuristics between the embeddings of the above transformations to construct relationships between the nodes.

In [20]:
from ragas.testset.transforms import default_transforms, apply_transforms

transformer_llm = generator_llm
embedding_model = generator_embeddings

default_transforms = default_transforms(documents=docs, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg, default_transforms)
kg

Applying HeadlinesExtractor:   0%|          | 0/17 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/20 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node


Applying SummaryExtractor:   0%|          | 0/31 [00:00<?, ?it/s]

Property 'summary' already exists in node 'f593d4'. Skipping!
Property 'summary' already exists in node '778a70'. Skipping!
Property 'summary' already exists in node 'efe16e'. Skipping!
Property 'summary' already exists in node 'f6a6e6'. Skipping!
Property 'summary' already exists in node '959da8'. Skipping!
Property 'summary' already exists in node '72443c'. Skipping!
Property 'summary' already exists in node '23ce63'. Skipping!
Property 'summary' already exists in node 'b32fb3'. Skipping!
Property 'summary' already exists in node 'bcb952'. Skipping!
Property 'summary' already exists in node 'c7a10d'. Skipping!
Property 'summary' already exists in node '22677b'. Skipping!
Property 'summary' already exists in node '6bf0cc'. Skipping!
Property 'summary' already exists in node '4362e7'. Skipping!
Property 'summary' already exists in node '553c21'. Skipping!


Applying CustomNodeFilter:   0%|          | 0/6 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/43 [00:00<?, ?it/s]

Property 'summary_embedding' already exists in node 'c7a10d'. Skipping!
Property 'summary_embedding' already exists in node 'bcb952'. Skipping!
Property 'summary_embedding' already exists in node '553c21'. Skipping!
Property 'summary_embedding' already exists in node 'efe16e'. Skipping!
Property 'summary_embedding' already exists in node 'f593d4'. Skipping!
Property 'summary_embedding' already exists in node '959da8'. Skipping!
Property 'summary_embedding' already exists in node 'b32fb3'. Skipping!
Property 'summary_embedding' already exists in node '72443c'. Skipping!
Property 'summary_embedding' already exists in node '4362e7'. Skipping!
Property 'summary_embedding' already exists in node 'f6a6e6'. Skipping!
Property 'summary_embedding' already exists in node '778a70'. Skipping!
Property 'summary_embedding' already exists in node '23ce63'. Skipping!
Property 'summary_embedding' already exists in node '22677b'. Skipping!
Property 'summary_embedding' already exists in node '6bf0cc'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

KnowledgeGraph(nodes: 40, relationships: 476)

We can save and load our knowledge graphs as follows.

In [21]:
kg.save("loan_data_kg.json")
loan_data_kg = KnowledgeGraph.load("loan_data_kg.json")
loan_data_kg

KnowledgeGraph(nodes: 40, relationships: 476)

Using our knowledge graph, we can construct a "test set generator" - which will allow us to create queries.

In [22]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=embedding_model, knowledge_graph=loan_data_kg)

However, we'd like to be able to define the kinds of queries we're generating - which is made simple by Ragas having pre-created a number of different "QuerySynthesizer"s.

Each of these Synthetsizers is going to tackle a separate kind of query which will be generated from a scenario and a persona.

In essence, Ragas will use an LLM to generate a persona of someone who would interact with the data - and then use a scenario to construct a question from that data and persona.

In [23]:
from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25),
]

#### ❓ Question #1:

What are the three types of query synthesizers doing? Describe each one in simple terms.

##### ✅ Answer: 
1. **Single-hop Specific Synthesizer**:  
   Generates direct, focused questions based on one context.  
   *Example: “What is a Pell Grant?”*

2. **Multi-hop Abstract Synthesizer**:  
   Creates high-level, reasoning-based questions using multiple documents.  
   *Example: “How does financial aid policy differ across program types?”*

3. **Multi-hop Specific Synthesizer**:  
   Produces detailed, multi-source questions targeting specific info.  
   *Example: “How do Appendices A and B guide Pell Grant disbursement?”*


Finally, we can use our `TestSetGenerator` to generate our testset!

In [24]:
testset = generator.generate(testset_size=10, query_distribution=query_distribution)
testset.to_pandas()

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,Whay are the Academic Calendars important for ...,"[Chapter 1 Academic Years, Academic Calendars,...",Chapter 1 explains that Academic Calendars are...,single_hop_specifc_query_synthesizer
1,What does 34 CFR 668.3(b) specify regarding we...,[Regulatory Citations Academic year minimums: ...,34 CFR 668.3(b) pertains to the weeks of instr...,single_hop_specifc_query_synthesizer
2,"As an academic program coordinator, how does t...",[Inclusion of Clinical Work in a Standard Term...,"Inclusion of clinical work in a standard term,...",single_hop_specifc_query_synthesizer
3,"According to federal regulations, how does the...",[Non-Term Characteristics A program that measu...,The payment period is applicable to all Title ...,single_hop_specifc_query_synthesizer
4,What is a Pell Grrant?,[both the credit or clock hours and the weeks ...,The Pell Grant is a type of federal financial ...,single_hop_specifc_query_synthesizer
5,How do the guidelines for calculating grant aw...,[<1-hop>\n\nboth the credit or clock hours and...,The guidelines specify that the amount of Pell...,multi_hop_abstract_query_synthesizer
6,How does the determination of successful progr...,[<1-hop>\n\nboth the credit or clock hours and...,The determination of successful program comple...,multi_hop_abstract_query_synthesizer
7,how disbursement timing in subscription progra...,[<1-hop>\n\nboth the credit or clock hours and...,The context explains that for the first two su...,multi_hop_abstract_query_synthesizer
8,How do the definitions of academic years in Vo...,"[<1-hop>\n\nChapter 1 Academic Years, Academic...",The definitions of academic years outlined in ...,multi_hop_specific_query_synthesizer
9,How do the definitions of academic years in Vo...,"[<1-hop>\n\nChapter 1 Academic Years, Academic...","According to the information in Volume 2, an a...",multi_hop_specific_query_synthesizer


### Abstracted SDG

The above method is the full process - but we can shortcut that using the provided abstractions!

This will generate our knowledge graph under the hood, and will - from there - generate our personas and scenarios to construct our queries.



In [28]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs[:20], testset_size=10)

Applying HeadlinesExtractor:   0%|          | 0/17 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/20 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node


Applying SummaryExtractor:   0%|          | 0/31 [00:00<?, ?it/s]

Property 'summary' already exists in node 'd94eaf'. Skipping!
Property 'summary' already exists in node '52c7f1'. Skipping!
Property 'summary' already exists in node '9c35e6'. Skipping!
Property 'summary' already exists in node 'd1f6ab'. Skipping!
Property 'summary' already exists in node 'd2732d'. Skipping!
Property 'summary' already exists in node '0845ea'. Skipping!
Property 'summary' already exists in node '7030eb'. Skipping!
Property 'summary' already exists in node 'e1bed7'. Skipping!
Property 'summary' already exists in node '347e2a'. Skipping!
Property 'summary' already exists in node 'de07f3'. Skipping!
Property 'summary' already exists in node 'd3db61'. Skipping!
Property 'summary' already exists in node '5f4d81'. Skipping!
Property 'summary' already exists in node '0555fc'. Skipping!
Property 'summary' already exists in node 'f54ac0'. Skipping!


...

Property 'summary_embedding' already exists in node 'd1f6ab'. Skipping!
Property 'summary_embedding' already exists in node 'd94eaf'. Skipping!
Property 'summary_embedding' already exists in node 'd3db61'. Skipping!
Property 'summary_embedding' already exists in node '9c35e6'. Skipping!
Property 'summary_embedding' already exists in node 'd2732d'. Skipping!
Property 'summary_embedding' already exists in node 'de07f3'. Skipping!
Property 'summary_embedding' already exists in node '347e2a'. Skipping!
Property 'summary_embedding' already exists in node '0845ea'. Skipping!
Property 'summary_embedding' already exists in node 'e1bed7'. Skipping!
Property 'summary_embedding' already exists in node '5f4d81'. Skipping!
Property 'summary_embedding' already exists in node 'f54ac0'. Skipping!
Property 'summary_embedding' already exists in node '52c7f1'. Skipping!
Property 'summary_embedding' already exists in node '0555fc'. Skipping!
Property 'summary_embedding' already exists in node '7030eb'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

In [29]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,How is the academic year defined for different...,"[Chapter 1 Academic Years, Academic Calendars,...",The academic year must be defined for every el...,single_hop_specifc_query_synthesizer
1,Chapter 3 what about clinical work in that cha...,[Inclusion of Clinical Work in a Standard Term...,Inclusion of clinical work in a standard term ...,single_hop_specifc_query_synthesizer
2,Is the Federal Work-Study program considered a...,[Non-Term Characteristics A program that measu...,"No, the Federal Work-Study (FWS) program is an...",single_hop_specifc_query_synthesizer
3,What is Volume 8 about?,[both the credit or clock hours and the weeks ...,The context discusses the disbursement of fede...,single_hop_specifc_query_synthesizer
4,differences academic year programs and how it ...,"[<1-hop>\n\nChapter 1 Academic Years, Academic...",Every program must have a defined academic yea...,multi_hop_abstract_query_synthesizer
5,How do the program requirements and exceptions...,[<1-hop>\n\nboth the credit or clock hours and...,The context explains that the determination of...,multi_hop_abstract_query_synthesizer
6,how do clock-hour and non-term credit-hour pro...,[<1-hop>\n\nboth the credit or clock hours and...,In clock-hour and non-term credit-hour program...,multi_hop_abstract_query_synthesizer
7,Effect of extra hours or weeks on grant and lo...,[<1-hop>\n\nboth the credit or clock hours and...,The context explains that completing additiona...,multi_hop_abstract_query_synthesizer
8,Chapter 2 include clinical work in standard te...,[<1-hop>\n\nInclusion of Clinical Work in a St...,Chapter 2 explains that clinical work conducte...,multi_hop_specific_query_synthesizer
9,How do the definitions of academic years and i...,[<1-hop>\n\nboth the credit or clock hours and...,The definitions of academic years and instruct...,multi_hop_specific_query_synthesizer


We'll need to provide our LangSmith API key, and set tracing to "true".

# 🤝 BREAKOUT ROOM #2

## Task 4: LangSmith Dataset

Now we can move on to creating a dataset for LangSmith!

First, we'll need to create a dataset on LangSmith using the `Client`!

We'll name our Dataset to make it easy to work with later.

In [30]:
from langsmith import Client

client = Client()

dataset_name = "Loan Synthetic Data"

langsmith_dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Loan Synthetic Data"
)

We'll iterate through the RAGAS created dataframe - and add each example to our created dataset!

> NOTE: We need to conform the outputs to the expected format - which in this case is: `question` and `answer`.

In [31]:
for data_row in dataset.to_pandas().iterrows():
  client.create_example(
      inputs={
          "question": data_row[1]["user_input"]
      },
      outputs={
          "answer": data_row[1]["reference"]
      },
      metadata={
          "context": data_row[1]["reference_contexts"]
      },
      dataset_id=langsmith_dataset.id
  )

## Basic RAG Chain

Time for some RAG!


In [32]:
rag_documents = docs

To keep things simple, we'll just use LangChain's recursive character text splitter!


In [33]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

We'll create our vectorstore using OpenAI's [`text-embedding-3-small`](https://platform.openai.com/docs/guides/embeddings/embedding-models) embedding model.

In [34]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

As usual, we will power our RAG application with Qdrant!

In [35]:
from langchain_community.vectorstores import Qdrant

vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Loan RAG"
)

In [36]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

To get the "A" in RAG, we'll provide a prompt.

In [37]:
from langchain.prompts import ChatPromptTemplate

RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

For our LLM, we will be using TogetherAI's endpoints as well!

We're going to be using Meta Llama 3.1 70B Instruct Turbo - a powerful model which should get us powerful results!

In [38]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini")

Finally, we can set-up our RAG LCEL chain!

In [39]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain.schema import StrOutputParser

rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | rag_prompt | llm | StrOutputParser()
)

In [40]:
rag_chain.invoke({"question" : "What kinds of loans are available?"})

'The kinds of loans available based on the context are:\n\n- Direct Subsidized Loans  \n- Direct Unsubsidized Loans  \n- Direct PLUS Loans (including student Federal PLUS Loans and parent Direct PLUS Loans)  \n- Subsidized and Unsubsidized Federal Stafford Loans (made under the FFEL Program before July 1, 2010)  \n- Federal SLS Loans (made under the FFEL Program before July 1, 2010)  \n- Federal PLUS Loans (made under the FFEL Program before July 1, 2010)  \n\nNote: No new loans have been made under the FFEL Program since July 1, 2010. Graduate or professional students are eligible only for Direct Unsubsidized Loans and Direct PLUS Loans, not Direct Subsidized Loans.'

## LangSmith Evaluation Set-up

We'll use OpenAI's GPT-4.1 as our evaluation LLM for our base Evaluators.

In [41]:
eval_llm = ChatOpenAI(model="gpt-4.1")

We'll be using a number of evaluators - from LangSmith provided evaluators, to a few custom evaluators!

In [42]:
from langsmith.evaluation import LangChainStringEvaluator, evaluate

qa_evaluator = LangChainStringEvaluator("qa", config={"llm" : eval_llm})

labeled_helpfulness_evaluator = LangChainStringEvaluator(
    "labeled_criteria",
    config={
        "criteria": {
            "helpfulness": (
                "Is this submission helpful to the user,"
                " taking into account the correct reference answer?"
            )
        },
        "llm" : eval_llm
    },
    prepare_data=lambda run, example: {
        "prediction": run.outputs["output"],
        "reference": example.outputs["answer"],
        "input": example.inputs["question"],
    }
)

empathy_evaluator = LangChainStringEvaluator(
    "criteria",
    config={
        "criteria": {
            "empathy": "Is this response empathetic? Does it make the user feel like they are being heard?",
        },
        "llm" : eval_llm
    }
)

#### 🏗️ Activity #2:

Highlight what each evaluator is evaluating.

- `qa_evaluator`:
- `labeled_helpfulness_evaluator`:
- `empathy_evaluator`:

#### ✅ Answer #:

- **`qa_evaluator`**:  
  Evaluates the **accuracy and completeness** of the generated answer compared to the reference answer. It checks if the model is factually correct and contextually aligned.

- **`labeled_helpfulness_evaluator`**:  
  Measures **how helpful** the answer is, often using labeled feedback (e.g., user or human ratings). It focuses on usefulness and clarity from a human-centric perspective.

- **`empathy_evaluator`**:  
  Evaluates whether the response shows **empathy and kindness** toward the user. It ensures the model’s tone is caring, polite, and user-sensitive.

## LangSmith Evaluation

In [43]:
evaluate(
    rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        empathy_evaluator
    ],
    metadata={"revision_id": "default_chain_init"},
)

View the evaluation results for experiment: 'proper-condition-76' at:
https://smith.langchain.com/o/2780c331-7464-47ba-a416-fa7fb91f6fdd/datasets/15d1d466-2273-4599-a2b2-6fe583f1ec2b/compare?selectedSessions=3d1f7a11-547c-46ea-b947-d51bcdf8474c




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.empathy,execution_time,example_id,id
0,How do the distinctions between Volume 8 and V...,"Based on the provided context:\n\n- Volume 7, ...",None,The context indicates that Volume 8 addresses ...,1,1,0,7.993188,7ee570f6-b256-4779-b01c-71036f295502,1d4a3552-d24c-436a-82b2-f21f0f27b152
1,How do Appendix A and Appendix B guide the dis...,I don't know.,None,Appendix B provides detailed guidance on disbu...,0,0,0,0.916631,25750ca2-fc90-4167-b108-0d390b800710,60e256c4-8311-4c35-9743-172d0bcf0ee2
2,How do the definitions of academic years and i...,"Based on the provided context, the definitions...",None,The definitions of academic years and instruct...,1,1,0,9.157889,5db27313-deea-42df-9ba6-6c9bf48067fe,ee5ccf22-bce7-4245-b2d0-97cedf13e64a
3,Chapter 2 include clinical work in standard te...,Based on the provided context:\n\nChapter 2 in...,None,Chapter 2 explains that clinical work conducte...,1,1,0,6.368748,58da12fa-8e45-4c43-a492-0a0fb594e881,b67f5143-c75c-441e-a2ee-26e0564cb0f0
4,Effect of extra hours or weeks on grant and lo...,Based on the provided context:\n\n- When a stu...,None,The context explains that completing additiona...,1,1,0,5.441446,426da852-23bd-40e3-a3d9-8e49330b0673,142fb3cb-2ce8-48d6-80f1-f1c62fdd14b3
5,how do clock-hour and non-term credit-hour pro...,Clock-hour and non-term credit-hour programs h...,None,In clock-hour and non-term credit-hour program...,0,0,0,3.066230,cffb0a9c-d5fc-4b1c-8762-7f3ae0844b1a,bf91b539-2c6f-44cd-81a2-c6a473e04989
6,How do the program requirements and exceptions...,"Based on the provided context, the determinati...",None,The context explains that the determination of...,1,1,0,21.116377,48e90ee9-f359-4926-bc5a-9332cbfbb388,56392bd1-056b-40d0-97da-82b645a47dcc
7,differences academic year programs and how it ...,The differences in academic year programs rela...,None,Every program must have a defined academic yea...,1,1,0,4.153870,c8da3325-0ac4-4085-9890-33c3b0c1b6fb,49480dbf-340a-4884-a651-f311a0d5ae06
8,What is Volume 8 about?,Volume 8 is about The Direct Loan Program. It ...,None,The context discusses the disbursement of fede...,0,0,0,1.147069,dcbaa019-e18a-4415-9bd4-1a64bc0d3169,678e59d6-a9e6-434c-b49a-ce505f66df21
9,Is the Federal Work-Study program considered a...,"Based on the context provided, the Federal Wor...",None,"No, the Federal Work-Study (FWS) program is an...",1,1,0,2.506509,2e2e6af9-5952-41dd-bfad-25687576e149,81740cd0-6aae-42e9-9739-dfc6e9b190c3


## Dope-ifying Our Application

We'll be making a few changes to our RAG chain to increase its performance on our SDG evaluation test dataset!

- Include a "dope" prompt augmentation
- Use larger chunks
- Improve the retriever model to: `text-embedding-3-large`

Let's see how this changes our evaluation!

In [44]:
EMPATHY_RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

You must answer the question using empathy and kindness, and make sure the user feels heard.

Context: {context}
Question: {question}
"""

empathy_rag_prompt = ChatPromptTemplate.from_template(EMPATHY_RAG_PROMPT)

In [45]:
rag_documents = docs

In [46]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

#### ❓Question #2:

Why would modifying our chunk size modify the performance of our application?

#### ✅ Answer #2:

Modifying the **chunk size** changes how the source documents are split before embedding and retrieval, which directly impacts performance:

1. **Too small chunk size**  
   → May result in **loss of context**, causing the retriever to return fragments that are incomplete or not meaningful.

2. **Too large chunk size**  
   → May reduce **retrieval accuracy**, since larger chunks might include irrelevant content, lowering embedding precision.

3. **Optimal chunk size**  
   → Balances context and specificity, leading to **better retrieval quality**, improved RAG response relevance, and higher evaluation scores (e.g., correctness, helpfulness).

The right chunk size ensures that query embeddings match the most relevant document pieces effectively.

In [47]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

#### ❓Question #3:

Why would modifying our embedding model modify the performance of our application?

#### ✅ Answer #3:

Modifying the **embedding model** affects how both queries and documents are represented in vector space, which directly impacts retrieval quality:

1. **Different embedding models** capture different semantic relationships. Some may better understand legal or academic language, others may generalize better across domains.

2. A **higher-quality embedding model** (e.g., one trained on your domain) improves **semantic similarity matching**, leading to more relevant chunks being retrieved.

3. Better retrieval leads to **more accurate and contextually aligned answers**, improving end-to-end RAG performance, including correctness, helpfulness, and empathy scores.

Therefore, choosing the right embedding model is crucial to ensure high-quality document retrieval and downstream response generation.

In [48]:
vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Loan Data for RAG"
)

In [49]:
retriever = vectorstore.as_retriever()

Setting up our new and improved DOPE RAG CHAIN.

In [50]:
empathy_rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | empathy_rag_prompt | llm | StrOutputParser()
)

Let's test it on the same output that we saw before.

In [51]:
empathy_rag_chain.invoke({"question" : "What kinds of loans are available?"})

"Thank you for your question. Based on the information you've provided, there are several types of loans available to students and their families to help cover the cost of attendance:\n\n1. **Direct Subsidized Loans** – These loans are for students with financial need, and the loan amount is based on the student's cost of attendance minus other aid received. Interest is subsidized by the government while the student is in school.\n\n2. **Direct Unsubsidized Loans** – These loans are available to students regardless of financial need and can be used to cover unmet financial need or replace the student's self-help aid.\n\n3. **Direct PLUS Loans** – These loans can be taken out by the parents of dependent students to cover the student's cost of attendance, assuming the parent meets eligibility requirements. There is no fixed loan limit, but the loan cannot exceed the total cost of attendance minus other aid.\n\nIt's also worth noting that if a dependent student's parent is unable to obtai

Finally, we can evaluate the new chain on the same test set!

In [52]:
evaluate(
    empathy_rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        empathy_evaluator
    ],
    metadata={"revision_id": "empathy_rag_chain"},
)

View the evaluation results for experiment: 'reflecting-attitude-7' at:
https://smith.langchain.com/o/2780c331-7464-47ba-a416-fa7fb91f6fdd/datasets/15d1d466-2273-4599-a2b2-6fe583f1ec2b/compare?selectedSessions=749ab24a-dab7-441b-af5a-0517dec91237




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.empathy,execution_time,example_id,id
0,How do the distinctions between Volume 8 and V...,Thank you for your thoughtful question. Naviga...,None,The context indicates that Volume 8 addresses ...,1,1,1,7.014275,7ee570f6-b256-4779-b01c-71036f295502,6dffa9ca-2d22-4a47-95cd-9ced0a1be1e9
1,How do Appendix A and Appendix B guide the dis...,Thank you for your thoughtful question. Based ...,None,Appendix B provides detailed guidance on disbu...,1,0,1,5.423001,25750ca2-fc90-4167-b108-0d390b800710,53309968-652b-4412-925e-716c322380da
2,How do the definitions of academic years and i...,Thank you for your thoughtful question. Naviga...,None,The definitions of academic years and instruct...,1,0,1,8.117611,5db27313-deea-42df-9ba6-6c9bf48067fe,37fe91f5-8f9c-4dd7-924f-31f33b16ec37
3,Chapter 2 include clinical work in standard te...,Thank you for your thoughtful question about h...,None,Chapter 2 explains that clinical work conducte...,1,1,1,6.117633,58da12fa-8e45-4c43-a492-0a0fb594e881,f131ac63-f716-4a97-a950-622cdf40b06d
4,Effect of extra hours or weeks on grant and lo...,Thank you for your thoughtful question about h...,None,The context explains that completing additiona...,1,0,1,7.017617,426da852-23bd-40e3-a3d9-8e49330b0673,385a0cbd-e826-4b1f-8fa8-aed5e43bbe5f
5,how do clock-hour and non-term credit-hour pro...,Thank you for your thoughtful question about h...,None,In clock-hour and non-term credit-hour program...,1,0,1,3.822508,cffb0a9c-d5fc-4b1c-8762-7f3ae0844b1a,a3f81f7c-c4b0-4254-92c8-aea2a58d2fe1
6,How do the program requirements and exceptions...,Thank you for asking such a thoughtful questio...,None,The context explains that the determination of...,1,1,1,9.778660,48e90ee9-f359-4926-bc5a-9332cbfbb388,4266fd72-79a6-4134-9ae6-3887b4fbfd9d
7,differences academic year programs and how it ...,Thank you for your thoughtful question about t...,None,Every program must have a defined academic yea...,1,1,1,6.113683,c8da3325-0ac4-4085-9890-33c3b0c1b6fb,80a4a3e3-8664-471f-b799-1289272f1d65
8,What is Volume 8 about?,Thank you for your question! Based on the cont...,None,The context discusses the disbursement of fede...,0,0,1,2.595478,dcbaa019-e18a-4415-9bd4-1a64bc0d3169,5166dfed-dfe8-4363-a48d-379064cb57d6
9,Is the Federal Work-Study program considered a...,I understand that navigating the details of fi...,None,"No, the Federal Work-Study (FWS) program is an...",1,1,1,4.137809,2e2e6af9-5952-41dd-bfad-25687576e149,a6ee5023-a8a1-4011-8e3a-6026322500f1


#### 🏗️ Activity #3:

Provide a screenshot of the difference between the two chains, and explain why you believe certain metrics changed in certain ways.

#### ✅ Answer #3:

![Screenshot 1](0E991D81-5105-4C28-B8ED-F6A86F6BAF25.jpeg)
![Screenshot 2](4193E983-D1A2-4645-A3D3-FF65B31A512E.jpeg)

##### 📸 Screenshot Comparison:
- ✅ Chain 1: `proper-condition-76` → Standard RAG prompt  
  - **Correctness**: 0.6667  
  - **Empathy**: 0.00  
  - **Helpfulness**: 0.80  

- ✅ Chain 2: `reflecting-attitude-7` → Empathetic prompt  
  - **Correctness**: 0.9167  
  - **Empathy**: 1.00  
  - **Helpfulness**: 0.50  

##### 📊 Why Metrics Changed:

1. **Correctness increased** (from 0.6667 to 0.9167):  
   - The empathetic prompt likely encouraged more detailed and thoughtful responses, improving factual alignment with the reference answers.

2. **Empathy increased** (from 0.00 to 1.00):  
   - As expected, the empathetic prompt used phrases like “Thank you for your thoughtful question” or “I understand…” which boosted the empathy evaluator score.

3. **Helpfulness decreased** (from 0.80 to 0.50):  
   - Despite sounding polite, some empathetic responses may have added verbosity or lacked directness, slightly reducing their perceived helpfulness or clarity.

##### 🎯 Summary:
Changing the prompt to reflect **empathy** improved **correctness** and **user tone**, but slightly reduced **helpfulness** due to more conversational responses. This highlights the tradeoff between emotionally supportive and concise answers in RAG applications.